## EEG-Based Classification of Major Depressive Disorder (MDD)

## Objective 
Build machine learning models to classify MDD vs healthy controls using EEG data

## Planned workflow
1. Inspect dataset structure and metadata
2. Define patient-level splits
3. Preprocess EEG recordings
4. Extract features
5. Train baseline models
6. Compare performance across conditions
7. Add and advanced extension 

In [2]:
%pip install mne

Note: you may need to restart the kernel to use updated packages.


In [3]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from pathlib import Path 
import mne

from sklearn.model_selection import GroupKFold, cross_validate
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC

In [4]:
# Project root and paths to data directories

PROJECT_ROOT = Path.cwd().parents[1]
RAW_DATA_DIR = PROJECT_ROOT / "MSSE-277b-final-project-" / "data" / "raw" / "nm000114"

print("Project root:", PROJECT_ROOT)
print("Exists?", PROJECT_ROOT.exists())
print("Raw data directory:", RAW_DATA_DIR)  
print("Exists?", RAW_DATA_DIR.exists())


Project root: /mnt/c/Users/AlexC/OneDrive/Desktop/berkeley_stuff/CHEM 277B
Exists? True
Raw data directory: /mnt/c/Users/AlexC/OneDrive/Desktop/berkeley_stuff/CHEM 277B/MSSE-277b-final-project-/data/raw/nm000114
Exists? True


In [5]:
# Helper function to load a single EDF file and return the raw data object

def get_eeg_file(subject_id: str, condition: str):
    return RAW_DATA_DIR / subject_id / "eeg" / f"{subject_id}_task-{condition}_eeg.edf"

In [6]:
# Test loading one file to ensure everything is working correctly

file_path = get_eeg_file("sub-HS1", "P300")

print(file_path)
print(file_path.exists())

raw = mne.io.read_raw_edf(file_path, preload=True)
print(raw)

/mnt/c/Users/AlexC/OneDrive/Desktop/berkeley_stuff/CHEM 277B/MSSE-277b-final-project-/data/raw/nm000114/sub-HS1/eeg/sub-HS1_task-P300_eeg.edf
True
Extracting EDF parameters from /mnt/c/Users/AlexC/OneDrive/Desktop/berkeley_stuff/CHEM 277B/MSSE-277b-final-project-/data/raw/nm000114/sub-HS1/eeg/sub-HS1_task-P300_eeg.edf...
Setting channel info structure...
Creating raw.info structure...


: 

In [ ]:
# Scan all files in the raw data directory to ensure we can access them

edf_files = sorted(RAW_DATA_DIR.glob("sub-*/*/*.edf"))

print(f"Number of EDF files found:", len(edf_files))
for file in edf_files[:5]:
    print(file)

Number of EDF files found: 181
D:\eeg-mdd-classification\data\raw\nm000114\sub-HS1\eeg\sub-HS1_task-eyesClosed_eeg.edf
D:\eeg-mdd-classification\data\raw\nm000114\sub-HS1\eeg\sub-HS1_task-eyesOpen_eeg.edf
D:\eeg-mdd-classification\data\raw\nm000114\sub-HS1\eeg\sub-HS1_task-P300_eeg.edf
D:\eeg-mdd-classification\data\raw\nm000114\sub-HS10\eeg\sub-HS10_task-eyesClosed_eeg.edf
D:\eeg-mdd-classification\data\raw\nm000114\sub-HS10\eeg\sub-HS10_task-eyesOpen_eeg.edf


In [7]:
# Parser cell to read all EDF files and extract data and labels for modeling

def infer_label_from_subject(subject_id: str) -> int:
    """
    Infer the label (0 or 1) from the subject ID.
    returns: 
        0 for healthy controls (sub-HS*)
        1 for MDD patients (sub-MDD*)
    """
    subject_upper = subject_id.upper()
    if "HS" in subject_upper:
        return 0
    elif "MDD" in subject_upper:
        return 1 
    else: 
        raise ValueError(f"Could not infer label from subject ID: {subject_id}")
    
def parse_condition_from_filename(filepath: Path) -> str:
    """
    Parse the condition from the filename
    Returns:
        - eyesClosed
        - eyesOpen
        - P300
    """

    name = filepath.name 
    if "task-eyesClosed" in name:
        return "eyesClosed"
    elif "task-eyesOpen" in name:
        return "eyesOpen"
    elif "task-P300" in name:
        return "P300"
    else:
        raise ValueError(f"Could not parse condition from filename: {name}")
        

In [8]:
# metadata list to hold all parsed information 

rows = []

for filepath in edf_files:
    subject_id = filepath.parts[-3]  # e.g. subHS1
    condition = parse_condition_from_filename(filepath)
    label = infer_label_from_subject(subject_id)

    rows.append({
        "patient_id": subject_id,
        "recording_id": filepath.stem,
        "label": label,
        "condition": condition,
        "filepath": str(filepath),
    })


# Create a DataFrame from the metadata
metadata_df = pd.DataFrame(rows)

print(metadata_df.head())
print("\nShape:", metadata_df.shape)
print("\nPatients:", metadata_df["patient_id"].nunique())
print("\nClass counts:")
print(metadata_df["label"].value_counts())
print("\nCondition counts:")
print(metadata_df["condition"].value_counts())
print(metadata_df.groupby("patient_id").size().value_counts())

NameError: name 'edf_files' is not defined

In [9]:
# save metadata to CSV for future reference
metadata_path = PROJECT_ROOT / "data" / "processed_features" / "metadata.csv"
metadata_df.to_csv(metadata_path, index=False)
print("Saved metadata to:", metadata_path)

NameError: name 'metadata_df' is not defined

In [10]:
feature_lengths = [len(r[0]) for r in records]

print("Unique feature lengths:", sorted(set(feature_lengths)))
print("Length counts:")
print(pd.Series(feature_lengths).value_counts().sort_index())

NameError: name 'records' is not defined

In [25]:
debug_rows = []

for _, row in metadata_df.iterrows():
    raw = mne.io.read_raw_edf(row["filepath"], preload=False, verbose=False)
    debug_rows.append({
        "patient_id": row["patient_id"],
        "condition": row["condition"],
        "filepath": row["filepath"],
        "n_channels": len(raw.ch_names),
        "channel_names": raw.ch_names
    })

debug_df = pd.DataFrame(debug_rows)

print(debug_df["n_channels"].value_counts())


n_channels
22    112
20     69
Name: count, dtype: int64


In [26]:
print("20-channel example:")
print(debug_df.loc[debug_df["n_channels"] == 20, "channel_names"].iloc[0])

print("\n22-channel example:")
print(debug_df.loc[debug_df["n_channels"] == 22, "channel_names"].iloc[0])


20-channel example:
['EEG Fp1-LE', 'EEG F3-LE', 'EEG C3-LE', 'EEG P3-LE', 'EEG O1-LE', 'EEG F7-LE', 'EEG T3-LE', 'EEG T5-LE', 'EEG Fz-LE', 'EEG Fp2-LE', 'EEG F4-LE', 'EEG C4-LE', 'EEG P4-LE', 'EEG O2-LE', 'EEG F8-LE', 'EEG T4-LE', 'EEG T6-LE', 'EEG Cz-LE', 'EEG Pz-LE', 'EEG A2-A1']

22-channel example:
['EEG Fp1-LE', 'EEG F3-LE', 'EEG C3-LE', 'EEG P3-LE', 'EEG O1-LE', 'EEG F7-LE', 'EEG T3-LE', 'EEG T5-LE', 'EEG Fz-LE', 'EEG Fp2-LE', 'EEG F4-LE', 'EEG C4-LE', 'EEG P4-LE', 'EEG O2-LE', 'EEG F8-LE', 'EEG T4-LE', 'EEG T6-LE', 'EEG Cz-LE', 'EEG Pz-LE', 'EEG A2-A1', 'EEG 23A-23R', 'EEG 24A-24R']


In [27]:
COMMON_CHANNELS = [
'EEG Fp1-LE', 'EEG F3-LE', 'EEG C3-LE', 'EEG P3-LE', 'EEG O1-LE',
 'EEG F7-LE', 'EEG T3-LE', 'EEG T5-LE', 'EEG Fz-LE', 'EEG Fp2-LE', 
 'EEG F4-LE', 'EEG C4-LE', 'EEG P4-LE', 'EEG O2-LE', 'EEG F8-LE', 
 'EEG T4-LE', 'EEG T6-LE', 'EEG Cz-LE', 'EEG Pz-LE', 'EEG A2-A1'
]

print("Number of common channels:", len(COMMON_CHANNELS))



Number of common channels: 20


In [30]:
# Step 1: Extract spectral features (e.g. power in different frequency bands) from each EEG recording

def extract_band_power(raw, COMMON_CHANNELS) -> np.ndarray:
    """
    Extract basic EEG band power features from the raw MNE objects.
    Returns a feature vector.
    """

    data_raw = raw.copy()
    data_raw.pick(COMMON_CHANNELS)
    
    # get data 
    data = data_raw.get_data()      # shape: (n_channels, n_time points)
    sfreq = data_raw.info['sfreq']  # sampling frequency (e.g. 256 Hz)

    # Define frequency bands 
    bands = {
        "delta": (1, 4),     # deep sleep
        "theta": (4, 8),     # drowsiness
        "alpha": (8, 13),    # relaxed 
        "beta":  (13, 30)    # active thinking 
    }

    all_features = []

    # Loop through each channel (ch) and convert signal to frequency domain using Welch's method to estimate power spectral density (psd)
    for ch in data:        
        psd, freqs = mne.time_frequency.psd_array_welch(
            ch,
            sfreq=sfreq,
            fmin=1,
            fmax=30,
            verbose=False
        )

        total_power = psd.sum()
        
        band_features = [
            (
                psd[(freqs >= fmin) & (freqs <= fmax)].mean()
            / total_power                                      # normalize by total power to get relative power in each band
            if total_power > 0 else 0                          # handle case where total power is zero to avoid division by zero
            )                           
            for (fmin, fmax) in bands.values()
            ]
    
        all_features.append(band_features)

    return np.array(all_features).flatten()



In [31]:
# Build feature matrix 

records = [
    (
        extract_band_power(
            mne.io.read_raw_edf(row["filepath"], preload=True, verbose=False),
            COMMON_CHANNELS
        ),
        row["label"],
        row["patient_id"]
    )
    for _, row in metadata_df.iterrows()
]

X = np.vstack([r[0] for r in records])
y = np.array([r[1] for r in records])
groups = np.array([r[2] for r in records])

print("X.shape:", X.shape)
print("y.shape:", y.shape)
print("Groups shape:", groups.shape)


X.shape: (181, 80)
y.shape: (181,)
Groups shape: (181,)
